In [1]:
#Step 1: Import the required libraries
import  pandas as pd
import numpy as np
import nltk  
import re 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.util import ngrams

In [2]:
#step 2:Download all corpus/Resources required for NLP
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
#Step 3: Create a sample dataset of complaints and their corresponding categories
data = {

    "complaint": [

        # ---------------- WATER ----------------

        "There is no water supply in my area",
        "Water pipeline is leaking near my house",
        "We have not received drinking water for three days",
        "The water supply is very irregular",
        "There is dirty water coming from the tap",
        "Water tank is empty and there is no supply",
        "Our locality has a serious water shortage",
        "The water pipeline needs immediate repair",

        # ---------------- ELECTRICITY ----------------

        "There is no electricity in our village",
        "Power supply has been interrupted since morning",
        "Electricity voltage is very low",
        "The transformer in our area is damaged",
        "We are facing frequent power cuts",
        "The electricity pole has fallen down",
        "There is no power supply for two days",
        "Please repair the damaged electricity transformer",

        # ---------------- ROADS ----------------

        "The road in our locality is completely damaged",
        "There are many potholes on the main road",
        "The road needs urgent repair",
        "The street road is full of potholes",
        "Road construction has been stopped",
        "The bridge on the road is damaged",
        "Vehicles cannot travel because of the bad road",
        "Please repair the broken road",

        # ---------------- HEALTH ----------------

        "The government hospital has no medicines",
        "There is no doctor available at the health centre",
        "The hospital staff is not available",
        "We need better medical facilities",
        "The health centre is closed during working hours",
        "Patients are waiting for a doctor",
        "The hospital does not have enough beds",
        "Please provide medicines at the hospital",

        # ---------------- SANITATION ----------------

        "Garbage has not been collected for many days",
        "There is garbage lying on the street",
        "The garbage collection service is very poor",
        "Our area is very dirty",
        "Waste is being dumped near our houses",
        "The garbage bin is overflowing",
        "There is no proper waste management",
        "Please clean the garbage from our locality"
    ],

    "category": [

        # Water
        "Water", "Water", "Water", "Water",
        "Water", "Water", "Water", "Water",

        # Electricity
        "Electricity", "Electricity", "Electricity", "Electricity",
        "Electricity", "Electricity", "Electricity", "Electricity",

        # Roads
        "Roads", "Roads", "Roads", "Roads",
        "Roads", "Roads", "Roads", "Roads",

        # Health
        "Health", "Health", "Health", "Health",
        "Health", "Health", "Health", "Health",

        # Sanitation
        "Sanitation", "Sanitation", "Sanitation", "Sanitation",
        "Sanitation", "Sanitation", "Sanitation", "Sanitation"
    ]
}


In [4]:
#Pipeline for NLP:
#Raw data-->Convert to lower case-->Remove punctuation-->Remove numbers-->Tokenize-->Remove stop Words-->
#Perform lemmatization-->Randomize the dataset -->Convert to TF-IDF vectorization --> Create a predictive model using Logistic Regression (Generate Unigram/Bigram/Trigram)
# --> Evaluate the model using accuracy, classification report, and confusion matrix

In [5]:
#Step 4: Create a DataFrame from the raw data
df = pd.DataFrame(data)
print(df.head())  # Display the first few rows of the DataFrame to verify the data
print(df.tail)

                                           complaint category
0                There is no water supply in my area    Water
1            Water pipeline is leaking near my house    Water
2  We have not received drinking water for three ...    Water
3                 The water supply is very irregular    Water
4           There is dirty water coming from the tap    Water
<bound method NDFrame.tail of                                             complaint     category
0                 There is no water supply in my area        Water
1             Water pipeline is leaking near my house        Water
2   We have not received drinking water for three ...        Water
3                  The water supply is very irregular        Water
4            There is dirty water coming from the tap        Water
5          Water tank is empty and there is no supply        Water
6           Our locality has a serious water shortage        Water
7           The water pipeline needs immediate repair        W

In [6]:
# Step Pre processing : Creating functions for steps 1 to 9 now
def preprocess_texts(texts):
    # Convert the text to lower case
    texts = texts.lower()
    # Remove punctuation
    texts = re.sub(r'[^a-zA-Z\s]', '', texts)  # Remove punctuation
    #Tokenize the text
    tokens = word_tokenize(texts)
    #Stopwords removal
    stop_words = set(stopwords.words('english'))
    tokens =[word for word in tokens if word.lower() not in stop_words] 
    #Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    clean_texts = ' '.join(tokens)
    return clean_texts

In [7]:
df['cleaned_complaint'] = df['complaint'].apply(preprocess_texts)
print(df.head()) 
print(df['cleaned_complaint'].head()) 
print(df['cleaned_complaint'].tail())  # Display the last few rows of the cleaned complaints

                                           complaint category  \
0                There is no water supply in my area    Water   
1            Water pipeline is leaking near my house    Water   
2  We have not received drinking water for three ...    Water   
3                 The water supply is very irregular    Water   
4           There is dirty water coming from the tap    Water   

                   cleaned_complaint  
0                  water supply area  
1  water pipeline leaking near house  
2  received drinking water three day  
3             water supply irregular  
4             dirty water coming tap  
0                    water supply area
1    water pipeline leaking near house
2    received drinking water three day
3               water supply irregular
4               dirty water coming tap
Name: cleaned_complaint, dtype: str
35                       area dirty
36          waste dumped near house
37          garbage bin overflowing
38          proper waste management


In [8]:
#Randomize the dataset
df = df.sample(frac=1).reset_index(drop=True)
print(df.head())  # Display the first few rows of the shuffled DataFrame

                                         complaint     category  \
0  Power supply has been interrupted since morning  Electricity   
1         There are many potholes on the main road        Roads   
2                The bridge on the road is damaged        Roads   
3           There is no electricity in our village  Electricity   
4              There is no water supply in my area        Water   

                        cleaned_complaint  
0  power supply interrupted since morning  
1                  many pothole main road  
2                     bridge road damaged  
3                     electricity village  
4                       water supply area  


In [9]:
df.category.unique()  # Display the unique categories in the dataset
df.category=df.category.str.lower()
#print(df.shape)  # Display the shape of the DataFrame after converting categories to lower case
#print(df.category.unique())  # Display the unique categories in the dataset after converting to lower case

In [10]:
print(df.category.unique())  # Display the unique categories in the dataset after converting to lower case

<StringArray>
['electricity', 'roads', 'water', 'health', 'sanitation']
Length: 5, dtype: str


In [11]:
#Step: convert sentiments into numeric labeles
df["category"]  = df["category"].map({"electricity": 0, "health": 1, "roads": 2, "sanitation": 3, "water": 4})
print(df.head())  # Display the first few rows of the DataFrame to verify the new 'label' column

                                         complaint  category  \
0  Power supply has been interrupted since morning         0   
1         There are many potholes on the main road         2   
2                The bridge on the road is damaged         2   
3           There is no electricity in our village         0   
4              There is no water supply in my area         4   

                        cleaned_complaint  
0  power supply interrupted since morning  
1                  many pothole main road  
2                     bridge road damaged  
3                     electricity village  
4                       water supply area  


In [12]:
#Step: Divinding  data into independent and dependent variables
X = df['cleaned_complaint']  # Independent variable (features)
Y = df['category']  # Dependent variable (target)

In [13]:
#Step: Split the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

In [14]:
#Create TF-IDF Vectorizer and transform the training and testing data
#ngram_range=  This means include both unigrams (single words) and bigrams (pairs of consecutive words) in the feature extraction process. 
#The min_df=1 ingrore extemenly reare words that appear in less than 1 document (in this case, since we have a small dataset, it will include all words).
#max_df tells ingore words which are coming more than 95% of the documents.This helps to avoid ovefitting the model to common words that do not carry much meaning.
Vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1,max_df= 0.96,stop_words='english')  # This means create a TF-IDF Vectorizer with unigrams and bigrams
X_train_tfidf = Vectorizer.fit_transform(X_train)
X_test_tfidf = Vectorizer.transform(X_test)

In [15]:
#Conver Tf-IDF matrix to the dataframe for better understanding and visualization
tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=Vectorizer.get_feature_names_out())
print(tfidf_df.head())  # Display the first few rows of the TF-IDF DataFrame to verify the transformation

   area  area dirty  available  available health  bad  bad road  bed  better  \
0   0.0         0.0        0.0               0.0  0.0       0.0  0.0     0.0   
1   0.0         0.0        0.0               0.0  0.0       0.0  0.0     0.0   
2   0.0         0.0        0.0               0.0  0.0       0.0  0.0     0.0   
3   0.0         0.0        0.0               0.0  0.0       0.0  0.0     0.0   
4   0.0         0.0        0.0               0.0  0.0       0.0  0.0     0.0   

   better medical  bin  ...  waste dumped  waste management     water  \
0             0.0  0.0  ...           0.0               0.0  0.000000   
1             0.0  0.0  ...           0.0               0.0  0.000000   
2             0.0  0.0  ...           0.0               0.0  0.000000   
3             0.0  0.0  ...           0.0               0.0  0.000000   
4             0.0  0.0  ...           0.0               0.0  0.246799   

   water coming  water day  water pipeline  water shortage  water supply  \
0   

In [16]:
model = LogisticRegression()  # Initialize the Logistic Regression model
model.fit(X_train_tfidf, Y_train)  # Train the model on the TF-IDF transformed training data

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [17]:
Y_pred = model.predict(X_test_tfidf)  # Predict the labels for the test data
print("Accuracy:", accuracy_score(Y_test, Y_pred))  # Calculate and print the accuracy of the model

Accuracy: 0.875


In [18]:
cm = confusion_matrix(Y_test, Y_pred)  # Compute the confusion matrix
print("Confusion Matrix:\n", cm)  # Print the confusion matrix

Confusion Matrix:
 [[2 0 0 1 0]
 [0 1 0 0 0]
 [0 0 2 0 0]
 [0 0 0 0 0]
 [0 0 0 0 2]]


In [20]:
new_complaints = [ "There is no drinking water in our village", 
                  "The electricity transformer is not working",
                  "There are huge potholes on our main road",
                  "The hospital has no doctor available",
                  "Garbage is not being collected from our area", 
                  "The water pipeline near my house is leaking", 
                  "We have frequent power cuts", 
                  "The road has been damaged due to heavy rain", 
                  "The government health centre needs more medicines", 
                  "Waste is being dumped near the school" ]

In [21]:
new_complaint_cleaned = [preprocess_texts(complaint) for complaint in new_complaints]  # Preprocess the new complaints
new_complaints_tfidf = Vectorizer.transform(new_complaint_cleaned)  # Transform the new complaints using the same TF-IDF vectorizer

In [22]:
new_predictions = model.predict(new_complaints_tfidf)  # Predict the sentiments for the new complaints
print("Predictions for new complaints:", new_predictions)  # Print the predictions for the new complaints

Predictions for new complaints: [4 0 2 1 3 4 0 2 1 3]


In [24]:
#df["category"]  = df["category"].map({"electricity": 0, "health": 1, "roads": 2, "sanitation": 3, "water": 4})
'''new_complaints = [ "There is no drinking water in our village", #4
                  "The electricity transformer is not working", #0
                  "There are huge potholes on our main road", #2
                  "The hospital has no doctor available", #1
                  "Garbage is not being collected from our area", #3
                  "The water pipeline near my house is leaking", #4
                  "We have frequent power cuts", #0
                  "The road has been damaged due to heavy rain", #2
                  "The government health centre needs more medicines", #1
                  "Waste is being dumped near the school" #3'''

'new_complaints = [ "There is no drinking water in our village", #4\n                  "The electricity transformer is not working", #0\n                  "There are huge potholes on our main road", #2\n                  "The hospital has no doctor available", #1\n                  "Garbage is not being collected from our area", #3\n                  "The water pipeline near my house is leaking", #4\n                  "We have frequent power cuts", #0\n                  "The road has been damaged due to heavy rain", #2\n                  "The government health centre needs more medicines", #1\n                  "Waste is being dumped near the school" #3'